<a href="https://colab.research.google.com/github/tanvi118/NLP_Lab/blob/main/NLP_Exp_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tanvi Vitankar, 500120658, exp-8

8.1 Analysing Meaning of Sentences

In [1]:
!pip install sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')

sentence1 = "The movie was excellent and I loved it."
sentence2 = "I really enjoyed the film."

embeddings = model.encode([sentence1, sentence2])

similarity = cosine_similarity(
    [embeddings[0]],
    [embeddings[1]]
)[0][0]

print("Sentence 1:", sentence1)
print("Sentence 2:", sentence2)
print("Semantic Similarity:", similarity)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence 1: The movie was excellent and I loved it.
Sentence 2: I really enjoyed the film.
Semantic Similarity: 0.85145193


In [4]:
sentences = [
    "The movie was excellent.",
    "The film was very good.",
    "The weather is hot today."
]

embeddings = model.encode(sentences)

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        score = cosine_similarity(
            [embeddings[i]],
            [embeddings[j]]
        )[0][0]

        print("\nSentence 1:", sentences[i])
        print("Sentence 2:", sentences[j])
        print("Similarity:", round(score, 4))


Sentence 1: The movie was excellent.
Sentence 2: The film was very good.
Similarity: 0.9277

Sentence 1: The movie was excellent.
Sentence 2: The weather is hot today.
Similarity: -0.0336

Sentence 1: The film was very good.
Sentence 2: The weather is hot today.
Similarity: -0.001


8.2 Text Summarization on IMDB Movie Reviews

In [23]:
import nltk
import numpy as np
from tensorflow.keras.datasets import imdb
from collections import Counter

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [24]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

print("Training Reviews:", len(x_train))
print("Testing Reviews:", len(x_test))

Training Reviews: 25000
Testing Reviews: 25000


In [25]:
word_index = imdb.get_word_index()

reverse_word_index = {
    value + 3: key
    for key, value in word_index.items()
}

reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(encoded_review):
    return " ".join(
        reverse_word_index.get(i, "?")
        for i in encoded_review
    )

review = decode_review(x_test[0])

print("Original Review:\n")
print(review)

print("\nSentiment:",
      "Positive" if y_test[0] == 1 else "Negative")

Original Review:

<START> please give this one a miss br br <UNK> <UNK> and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allowed this one on his plate he almost seemed to know this wasn't going to work out and his performance was quite <UNK> so all you madison fans give this a miss

Sentiment: Negative


In [26]:
def summarize_text(text, num_sentences=3):

    sentences = sent_tokenize(text)

    if len(sentences) <= num_sentences:
        return text

    stop_words = set(stopwords.words("english"))

    words = word_tokenize(text.lower())

    words = [
        word for word in words
        if word.isalnum() and word not in stop_words
    ]

    word_frequencies = Counter(words)

    sentence_scores = {}

    for sentence in sentences:
        sentence_words = word_tokenize(sentence.lower())

        score = 0

        for word in sentence_words:
            if word in word_frequencies:
                score += word_frequencies[word]

        sentence_scores[sentence] = score

    ranked_sentences = sorted(
        sentence_scores,
        key=sentence_scores.get,
        reverse=True
    )

    selected = ranked_sentences[:num_sentences]

    # Preserve original sentence order
    summary = [
        sentence for sentence in sentences
        if sentence in selected
    ]

    return " ".join(summary)

In [27]:
summary = summarize_text(review, num_sentences=3)

print("ORIGINAL REVIEW:")
print(review)

print("\n" + "=" * 60)

print("\nSUMMARY:")
print(summary)

ORIGINAL REVIEW:
<START> please give this one a miss br br <UNK> <UNK> and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allowed this one on his plate he almost seemed to know this wasn't going to work out and his performance was quite <UNK> so all you madison fans give this a miss


SUMMARY:
<START> please give this one a miss br br <UNK> <UNK> and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allowed this one on his plate he almost seemed to know this wasn't going to work out and his performance was quite <UNK> so all you madison fans give this a miss


In [28]:
for i in range(5):

    review = decode_review(x_test[i])

    summary = summarize_text(review, num_sentences=3)

    print("\n" + "=" * 70)
    print("REVIEW", i + 1)
    print("=" * 70)

    print("\nOriginal Review:")
    print(review[:700], "...")

    print("\nSentiment:",
          "Positive" if y_test[i] == 1 else "Negative")

    print("\nSummary:")
    print(summary)


REVIEW 1

Original Review:
<START> please give this one a miss br br <UNK> <UNK> and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allowed this one on his plate he almost seemed to know this wasn't going to work out and his performance was quite <UNK> so all you madison fans give this a miss ...

Sentiment: Negative

Summary:
<START> please give this one a miss br br <UNK> <UNK> and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allowed this one on his plate he almost seemed to know this wasn't going to work out and his performance was quite <UNK> so all you madison fans give this a miss

REVIEW 2

Original Review:
<START> this film requires a lot of patience because it focuses on mood and character development the plot is very simple and many of the scenes take place on the same set in frances <UNK> the sandy dennis chara